# Exploratory Analysis - Titanic Survivors

## Validating Hypothesis #3

### **Analyzing Text with Feature Engineering**

<img src="https://i.imgur.com/gfVeG5l.gif" width="500"/>


### Business Goal

> Identify the factors that led some Titanic passengers to survive and others not.
>
> `Reason`: make future cruises safer.

### Notebook Goal

> 1. Manipulate Text
> 2. Visualize Categorical Data (Histogram)

### 🗺️ Mapped Hypotheses

> 🔵: Hypotheses covered in this notebook
>
> ✔️: Hypotheses already validated

Passengers more likely to survive:
1. From higher classes (`Pclass` 1 > 2 > 3) ✔️
2. With more expensive tickets (`Fare`) ✔️
3. With distinguished titles (`Name`) 🔵
4. Aged 20-35 (`Age`)
5. Female (`Sex`)
6. With small families (`Parch` & `SibSp`)

Bonus:
7. Larger families are from lower classes
8. Higher classes have older passengers

![Hipóteses](https://c.tenor.com/Nq-PcKRJUFsAAAAC/bart-simpson-reading.gif)



### Install external dependencies

In [ ]:
!pip install --force-reinstall git+https://github.com/murilobellatini/eda

### Import Libs

In [ ]:
# custom libs
from src.paths import PROCESSED_DATA_PATH_TITANIC
from src.io import load_df_from_url, mount_gdrive
from src.stats import compare2groups

# visualization libs
import matplotlib.pyplot as plt
import seaborn as sns

# processing libs
import pandas as pd
import numpy as np
import os


sns.set()

### Load DataFrame

In [ ]:
# mount Google Drive and load `processed/train_aula_06.csv`
mount_gdrive()
df = pd.read_csv(f'{PROCESSED_DATA_PATH_TITANIC}/train_aula_06.csv', index_col=0)
df

# Data Preparation

How will we define a title column?

<img src="https://media2.giphy.com/media/1NWend5SzXVeQ0CPBw/giphy.gif" width=500>

> *Always be prepared*

In [ ]:
# show the "Name" column
df.Name

# Let's manipulate text!

<img src="https://64.media.tumblr.com/5bf54838a935421531250333b12b6a06/tumblr_nf1udwp9dj1so28u7o1_500.gifv" width=500>

In [ ]:
# split "Name" into words (at its spaces)
df.Name.str.split(" ")

In [ ]:
# store the words of name in column "Name_list"
df['Name_list'] = df.Name.str.split()

# explode column "Name_list" as Tidy Data
df.explode('Name_list')

# 💡 Technical Tip!!

> `Tidy Data` is amazing!
>
> We just restructured our data into this format.
>
> It follows the triad 3️⃣ below:
>
> 1. Each column is a variable
> 2. Each row is an observation
> 3. Each cell has a single value
> 
> `Reason`: Much easier to analyze the data and extract metrics.
>
> ⚠️ `Warning`: Uses more memory/storage because it is a denormalized format (redundant data).

<img src="https://i.pinimg.com/originals/41/5b/fd/415bfd2c979b0f4c671a3a133e441785.gif" width=500/>

> *Keeping the house __tidy__ (organized)!*

In [ ]:
# count the words of column "Name_list"
df.Name_list.explode().value_counts()

In [ ]:
# show the 20 most frequent words of column "Name_list"
df.Name_list.explode().value_counts().head(20)

In [ ]:
# store the words of column "Name_list" in variable `tmp`
tmp = df.Name_list.explode()
tmp

In [ ]:
# count the words of variable `tmp` that contain a period "."
mask = tmp.str.contains('\.')
tmp[mask].value_counts()

In [ ]:
# extract from column "Name" the words that end with a period "."
# and store them in column "Title"
df['Title'] = df.Name.str.extract('(\w*\.)')
df.head()

# 💡 Technical Tip!!

> We just built a Regular Expression (or RegEx).
>
> They were made to manipulate text.
>
> `Reason`:
>
> > They help find complex text patterns.
>
> ⚠️ `Warning`:
> > RegEx takes a lot of computing effort. If you need performance, look for cheaper ways.
>
<img src="https://c.tenor.com/NGFJ4ILyBtcAAAAC/spongebob-reading.gif" width=500/>

> *Finding what I need... Without reading!*

In [ ]:
# count the values of column "Title"
df.Title.value_counts()

In [ ]:
# remove column "Name_list"
df = df.drop('Name_list', axis=1)

# 💡 Technical Tip!!

> Remove redundant columns whenever you can.
>
> `Reason`:
>
> 1. Stay organized
> 2. Avoid confusion
> 3. Save memory (large datasets)
>
> ⚠️ `Warning`: When you are doing Feature Engineering this can be a bad idea.
>
<img src="https://media2.giphy.com/media/du9tUy0lntYsqGF2Ye/200.gif" width=500/>

> *Deleting.*

# Let's trim this list down?

<img src="https://ksr-ugc.imgix.net/assets/028/543/333/22d975b407a492ab0222c5febac6d607_original.gif?ixlib=rb-4.0.2&w=680&fit=max&v=1585256901&auto=format&gif-q=50&q=92&s=cb6d034ecbb2282bcfc68fcbb85f17a5" width=500>

In [ ]:
# select into a list/array all the titles except the first 5 (top 4)
uncommon_titles = df.Title.value_counts().index[4:].values
uncommon_titles

In [ ]:
#  filter `df` to uncommon titles only
mask = df.Title.isin(uncommon_titles)
df[mask].Title.value_counts()

In [ ]:
# replace uncommon titles with "Uncommon Title"
# and count the new titles
df.loc[mask, "Title"] = 'Uncommon Title'
df.Title.value_counts()

# 💡 Technical Tip!!

> We just did `Feature Engineering`!
>
> It is the process of extracting useful information from raw data.
>
> `Reason`
>
> 1. Generate variables (features) for Machine Learning
> 2. Generate useful KPIs for decision making (Analytics)
> 3. Bring business know-how into the data

<img src="https://thumbs.gfycat.com/LongNaiveAmericanpainthorse-size_restricted.gif" width=500>

> *Feature Engineering is the music of Data Engineers!*

# Validating Hypothesis #3

> `Hypothesis`: Passengers with distinguished titles (`Name`) are more likely to survive

With the data prepared, let's get to action!

<img src="https://media0.giphy.com/media/dr6toZX3D1O8/200.gif" width=500/>

> Go!!!!!

In [ ]:
# compute the percentage of survivors by title
survived_title = df.groupby('Title').Survived.mean().rename('Survived_perc').sort_values(ascending=False)
survived_title 

In [ ]:
# compute statistics (count, mean, standard deviation, etc) of survivors by title
tmp = df.groupby('Title').Survived.describe().sort_values('mean', ascending=False)
tmp

# Let's make it more visual?

> The histogram is perfect for that!
>
> It shows the distribution of your categorical data.
>
> It is a visual `.value_counts()`
>
> Great for a quick idea of what your `Categorical Data` looks like!


<img src="https://media.giphy.com/media/xUPGcqaVH1cDeKZTBS/giphy.gif" width=500>

>
> *I hate counting...*

In [ ]:
# visualize the titles as a histogram
_ = sns.histplot(data=df, y='Title')

In [ ]:
# make a bar chart with the means of survivors
_ = (100*tmp['mean']).plot(
    kind='bar', table=True, legend=False,
    title='Distribuição de Sobreviventes por Título',
    ylabel='% Sobreviventes', xlabel='\n\nTítulo do Passageiro',
    fontsize=14, xticks=(),
    figsize=(14,7)
    )

In [ ]:
# show titles by sex
df.groupby(['Sex', 'Title']).size()

In [ ]:
# validate statistically
compare2groups(
    df[df.Title=='Mrs.'].Survived,
    df[df.Title=='Miss.'].Survived
    )

In [ ]:
# validate statistically
compare2groups(
    df[df.Title=='Mrs.'].Survived,
    df[df.Title=='Master.'].Survived
    )

# 👌 Insight / Conclusion

Hypothesis #3 validated. ✔️

> Passengers with certain titles are more likely to survive:
> > Married women (`Mrs.`) & single women (`Miss.`) > masters (`Master`)
>
> ⚠️ `Note`
> > The samples of `Mrs.` and `Miss.` show no statistical difference.
> >
> > But `Mrs` and `Master` do!
> >
> > Still, the hypothesis remains validated.

<img src="https://media1.giphy.com/media/l1BTt8fGSc207loYhY/200.gif" width=500>

> *I love my titles*

# 💡 Technical Tip!!

> Master the basics first.
>
> 1. Generate statistics (`.mean()`, `.describe()`)
> 2. Do simple manipulations (`.apply`)
> 3. Make simple charts (`.plot()` with histograms, boxplots)
> 4. Use p-value in your validations
>
> ⚠️ `Warning`: More important than all of this... Understand what you are doing!

<img src="https://media0.giphy.com/media/xUOwG5IshvzNMdzR72/200.gif" width=500>

> *Basics. Basics. Basics.*

In [ ]:
# export the dataset with the new feature `Title`
if not os.path.exists(PROCESSED_DATA_PATH_TITANIC):
  os.makedirs(PROCESSED_DATA_PATH_TITANIC)
  
df.to_csv(f"{PROCESSED_DATA_PATH_TITANIC}/train_aula_07.csv") 

## Summary

In this lesson we learned:

1. Manipulate text (RegEx and `.str`)
2. Structure data for Analytics (`Tidy Data`)
3. The concept of `Feature Engineering`
4. How to validate one more hypothesis in practice

<img src="https://media1.giphy.com/media/3ov9k01Y5IKizNmC7S/200.gif" width="500"/>


# Done!

That's all for this lesson...

<img src="https://c.tenor.com/IAUvvrUY7zQAAAAM/done-spongebob.gif" width="500"/>

> *Done*